In [ ]:
from FIT_python.pipeline_sex.data_import_utils import load_raw_files
from FIT_python.config import RAW_DIR
raw_dfs = load_raw_files(RAW_DIR)
raw_dfs

In [ ]:
# %% [code]
from pathlib import Path
import pandas as pd
import numpy as np
import ast
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import ParameterSampler

from FIT_python.pipeline_sex.pipeline_wrapper_sex import PipelineWrapper
from FIT_python.config import RESULTS_DATA_DIR

# 1) Prepare (Clean → Split → Summary)
prep = PipelineWrapper(
    model_keys=["log_reg"],  # Dummy
    impute_method=None,
    outlier_method=None,
    scaler_method=None,
    reduce_pre_method=None,
    reduce_post_method=None,
    fs_method=None,
    fs_k=None,
)
prep.prepare()

# 2) Output-Dateien
# Ergebnisse-Basis
res_dir = Path(RESULTS_DATA_DIR) / "sex_model"

# Stelle sicher, dass der Ordner da ist
res_dir.mkdir(parents=True, exist_ok=True)

# Datei-Pfade
raw_csv  = res_dir / "raw_results.csv"
best_csv = res_dir / "best_results.csv"

# Alte Dateien löschen
if raw_csv.exists():  raw_csv.unlink()
if best_csv.exists(): best_csv.unlink()

# 3) Parametrierräume
# Verfügbare Model-Keys aus MODELS
model_keys = [
    "logreg_l2", "logreg_l1",
    "rf_small", "rf_med", "rf_large",
    "et_med", "et_sm",
    "knn_3", "knn_5", "knn_7",
    "svm_linear", "svm_rbf",
    "xgb_std", "xgb_hist",
    "lgbm_std", "lgbm_md10",
    "catb_std", "catb_fast","lda",
]

param_dist = {
    "impute_method":      [None],
    "outlier_method":     [None, "clip", "zscore"],
    "scaler_method":      [None, "standard", "robust"],
    "reduce_pre_method":  [None, "pca", "umap"],
    "reduce_post_method": [None, "pca", "umap"],
    "fs_method":          [None, "forward", "random_forest", "variance", "univariate", "lasso"],
    "fs_k":               [0, 1, 2, 3, 4, 5, 6, 7, 10, 20, 50, 100],
    "model_key":          model_keys,
}
# nur 100 zufällige Kombinationen
grid = list(ParameterSampler(param_dist, n_iter=10, random_state=42))


# Hilfsfunktion für NaN→None
def clean_none(x):
    return None if pd.isna(x) else x

def run_trial(params):
    """Ein einzelner Trial, gibt DataFrame mit raw_results zurück."""
    # a) Parameter in Python-Typen
    impute   = clean_none(params["impute_method"])
    outlier  = clean_none(params["outlier_method"])
    scaler   = clean_none(params["scaler_method"])
    red_pre  = clean_none(params["reduce_pre_method"])
    red_post = clean_none(params["reduce_post_method"])
    fs_meth  = clean_none(params["fs_method"])
    fs_k_raw = params["fs_k"]
    fs_k     = None if (pd.isna(fs_k_raw) or fs_k_raw == 0) else int(fs_k_raw)
    model    = clean_none(params["model_key"])

    # b) Pipeline trainieren
    pw = PipelineWrapper(
        model_keys=[model],
        impute_method=     impute,
        outlier_method=    outlier,
        scaler_method=     scaler,
        reduce_pre_method= red_pre,
        reduce_post_method=red_post,
        fs_method=         fs_meth,
        fs_k=              fs_k,
    )
    df_raw = pw.train()

    # c) classification_report evtl. parsen
    df_raw["classification_report"] = df_raw["classification_report"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    # d) Balanced Accuracy berechnen
    df_raw["balanced_accuracy"] = df_raw["classification_report"].apply(
        lambda cr: cr["macro avg"]["recall"]
    )
    # e) Metadaten anhängen
    for k, v in params.items():
        df_raw[k] = v

    return df_raw

# 4) Parallel ausführen
results = Parallel(n_jobs=8, verbose=10)(
    delayed(run_trial)(p) for p in grid
)

# 5) Alle Ergebnisse zusammenführen und CSV schreiben
df_all = pd.concat(results, ignore_index=True)
df_all.to_csv(raw_csv, index=False)

# 6) Besten Parameter-Satz bestimmen und speichern
# (oder alternativ: df_all.groupby(...).mean().idxmax())

print("Fertig! Siehe:")
print(f" • {raw_csv}")


In [ ]:
import pandas as pd
from joblib import load

# Pfade definieren
model_path = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/sex_models_best/eurasian_otter.joblib"
data_path  = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/raw/Eurasian Otter.csv"
output_csv = "C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data/eurasian_otter_predictions.csv"

# Modell laden
clf = load(model_path)

# Daten laden und Spaltennamen anpassen (Punkte, Bindestriche → Unterstriche; T → t)
df = pd.read_csv(data_path)
df.columns = [
    col.replace('.', '_')
       .replace('-', '_')
       .replace('T', 't')
    for col in df.columns
]

# Direkt Vorhersagen aus der Pipeline holen
df["predicted_sex"]     = clf.predict(df)
probas = clf.predict_proba(df)
df["predicted_proba_f"] = probas[:, 0]
df["predicted_proba_m"] = probas[:, 1]

# Ergebnisse als CSV speichern
df.to_csv(output_csv, index=False)
print(f"Ergebnisse gespeichert in: {output_csv}")
